# BERT vs DistilBERT vs RoBERTa on SST-2
**Zhiren Xia — STATS 507 Final Project**

In [ ]:
import sys
!{sys.executable} -m pip install transformers datasets evaluate torch scikit-learn matplotlib numpy pandas accelerate

In [ ]:
import torch
import transformers
import datasets
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS")
else:
    device = torch.device("cpu")
    print("Using CPU")

## Data Loading

In [ ]:
dataset = load_dataset("glue", "sst2")
print(dataset)
print(f"\nTrain: {len(dataset['train'])}")
print(f"Val: {len(dataset['validation'])}")

In [ ]:
for i in range(5):
    ex = dataset["train"][i]
    label = "pos" if ex["label"] == 1 else "neg"
    print(f"[{label}] {ex['sentence']}")

In [ ]:
train_labels = dataset["train"]["label"]
val_labels = dataset["validation"]["label"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts_train = [train_labels.count(0), train_labels.count(1)]
axes[0].bar(["Negative", "Positive"], counts_train, color=["#E74C3C", "#2ECC71"])
axes[0].set_title("Training Set")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts_train):
    axes[0].text(i, v + 200, str(v), ha="center", fontweight="bold")

counts_val = [val_labels.count(0), val_labels.count(1)]
axes[1].bar(["Negative", "Positive"], counts_val, color=["#E74C3C", "#2ECC71"])
axes[1].set_title("Validation Set")
axes[1].set_ylabel("Count")
for i, v in enumerate(counts_val):
    axes[1].text(i, v + 5, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sentence_lengths = [len(s.split()) for s in dataset["train"]["sentence"]]

plt.figure(figsize=(8, 4))
plt.hist(sentence_lengths, bins=50, color="#3498DB", edgecolor="white")
plt.xlabel("Sentence Length (words)")
plt.ylabel("Count")
plt.title("Sentence Length Distribution (Train)")
plt.axvline(x=np.mean(sentence_lengths), color="red", linestyle="--",
            label=f"Mean: {np.mean(sentence_lengths):.1f}")
plt.axvline(x=np.percentile(sentence_lengths, 95), color="orange", linestyle="--",
            label=f"95th pct: {np.percentile(sentence_lengths, 95):.0f}")
plt.legend()
plt.tight_layout()
plt.savefig("sentence_lengths.png", dpi=150, bbox_inches="tight")
plt.show()

## Tokenization

In [ ]:
MODEL_NAMES = {
    "BERT":       "bert-base-uncased",
    "DistilBERT": "distilbert-base-uncased",
    "RoBERTa":    "roberta-base",
}

def tokenize_function(examples, tokenizer):
    return tokenizer(
        examples["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

## Training

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

all_results = {}
all_trainers = {}

def train_model(model_name, model_path):
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    tokenized_train = dataset["train"].map(
        lambda x: tokenize_function(x, tokenizer), batched=True
    )
    tokenized_val = dataset["validation"].map(
        lambda x: tokenize_function(x, tokenizer), batched=True
    )

    training_args = TrainingArguments(
        output_dir=f"./results/{model_name}",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        logging_steps=100,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )

    start_time = time.time()
    train_result = trainer.train()
    train_time = time.time() - start_time

    # workaround for NotebookProgressCallback bug
    try:
        trainer.pop_callback(transformers.integrations.notebook.NotebookProgressCallback)
    except Exception:
        pass

    eval_result = trainer.evaluate()

    log_history = trainer.state.log_history
    epoch_metrics = [
        {"epoch": e["epoch"], "eval_accuracy": e["eval_accuracy"], "eval_loss": e["eval_loss"]}
        for e in log_history if "eval_accuracy" in e
    ]
    train_losses = [(e["step"], e["loss"]) for e in log_history if "loss" in e and "eval_loss" not in e]

    results = {
        "accuracy": eval_result["eval_accuracy"],
        "train_time": train_time,
        "train_loss": train_result.training_loss,
        "eval_loss": eval_result["eval_loss"],
        "num_params": sum(p.numel() for p in model.parameters()),
        "epoch_metrics": epoch_metrics,
        "train_losses": train_losses,
    }

    print(f"Accuracy: {eval_result['eval_accuracy']*100:.1f}%")
    print(f"Time: {train_time/60:.1f} min")

    return results, trainer

In [ ]:
all_results["BERT"], all_trainers["BERT"] = train_model("BERT", MODEL_NAMES["BERT"])

In [ ]:
all_results["DistilBERT"], all_trainers["DistilBERT"] = train_model("DistilBERT", MODEL_NAMES["DistilBERT"])

In [ ]:
all_results["RoBERTa"], all_trainers["RoBERTa"] = train_model("RoBERTa", MODEL_NAMES["RoBERTa"])

## Results

In [ ]:
summary_data = []
for name, result in all_results.items():
    summary_data.append({
        "Model": name,
        "Accuracy (%)": f"{result['accuracy']*100:.2f}",
        "Parameters (M)": f"{result['num_params']/1e6:.1f}",
        "Training Time (min)": f"{result['train_time']/60:.1f}",
        "Train Loss": f"{result['train_loss']:.4f}",
        "Eval Loss": f"{result['eval_loss']:.4f}",
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
summary_df.to_csv("model_comparison.csv", index=False)

## Visualization

In [ ]:
models = list(all_results.keys())
colors = ["#4472C4", "#ED7D31", "#70AD47"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

accuracies = [all_results[m]["accuracy"] * 100 for m in models]
bars = axes[0].bar(models, accuracies, color=colors, width=0.5)
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("(a) Validation Accuracy")
axes[0].set_ylim(min(accuracies) - 3, max(accuracies) + 2)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                 f"{acc:.1f}%", ha="center", fontweight="bold")

times = [all_results[m]["train_time"] / 60 for m in models]
bars = axes[1].bar(models, times, color=colors, width=0.5)
axes[1].set_ylabel("Time (minutes)")
axes[1].set_title("(b) Training Time")
for bar, t in zip(bars, times):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 f"{t:.0f}m", ha="center", fontweight="bold")

params = [all_results[m]["num_params"] / 1e6 for m in models]
bars = axes[2].bar(models, params, color=colors, width=0.5)
axes[2].set_ylabel("Parameters (millions)")
axes[2].set_title("(c) Model Size")
for bar, p in zip(bars, params):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f"{p:.0f}M", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for model_name, color in zip(models, colors):
    result = all_results[model_name]

    if result["train_losses"]:
        steps, losses = zip(*result["train_losses"])
        axes[0].plot(steps, losses, label=model_name, color=color, alpha=0.8)

    if result["epoch_metrics"]:
        epochs = [m["epoch"] for m in result["epoch_metrics"]]
        accs = [m["eval_accuracy"] for m in result["epoch_metrics"]]
        axes[1].plot(epochs, accs, marker="o", label=model_name, color=color, linewidth=2)

axes[0].set_xlabel("Training Steps")
axes[0].set_ylabel("Training Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
best_model_name = max(all_results, key=lambda k: all_results[k]["accuracy"])
best_trainer = all_trainers[best_model_name]

best_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAMES[best_model_name])
tokenized_val = dataset["validation"].map(
    lambda x: tokenize_function(x, best_tokenizer), batched=True
)

predictions = best_trainer.predict(tokenized_val)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

cm = confusion_matrix(true_labels, pred_labels)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

print(classification_report(true_labels, pred_labels, target_names=["Negative", "Positive"]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for i, model_name in enumerate(models):
    result = all_results[model_name]
    ax.scatter(
        result["num_params"] / 1e6,
        result["accuracy"] * 100,
        s=result["train_time"] / 5,
        c=colors[i],
        label=f"{model_name} ({result['train_time']/60:.0f}min)",
        edgecolors="black", linewidth=1, alpha=0.8, zorder=5,
    )

ax.set_xlabel("Parameters (millions)")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy vs. Model Size (bubble size = training time)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("efficiency_accuracy.png", dpi=200, bbox_inches="tight")
plt.show()

## Error Analysis

In [ ]:
wrong_indices = np.where(pred_labels != true_labels)[0]
val_sentences = dataset["validation"]["sentence"]

print(f"Misclassified: {len(wrong_indices)} / {len(true_labels)}\n")

for idx in wrong_indices[:10]:
    true = "pos" if true_labels[idx] == 1 else "neg"
    pred = "pos" if pred_labels[idx] == 1 else "neg"
    print(f"[true={true}, pred={pred}] {val_sentences[idx]}")